# 📅 Exercício Extra 3: O Robô "Guarda-Costas" 👤🛡️

Neste exercício, vamos carregar um modelo de Inteligência Artificial pré-treinado pelo OpenCV chamado **Haar Cascade**. Este modelo serve para detetar rostos humanos instantaneamente.

O robô vai medir o tamanho do teu rosto no ecrã para adivinhar se estás perto ou longe:
* Rosto **Grande** no ecrã = Estás muito perto!
* Rosto **Pequeno** no ecrã = Estás longe!

### 🎯 O Teu Objetivo
Aparecer à frente da câmara do robô e ver o sistema detetar o teu rosto com um quadrado azul e calcular a tua proximidade.

### 🛠️ Instruções
1. Executa a célula.
2. Olha para a câmara do JetRacer. Afasta-te e aproxima-te para ver o tamanho do quadrado a mudar em tempo real.

In [ ]:
import cv2
import ipywidgets as widgets
from IPython.display import display
from jetcam.csi_camera import CSICamera
import time

print("--- SISTEMA GUARDA-COSTAS ACTIVO ---")

# Carrega o ficheiro xml de IA padrão do OpenCV para rostos
classificador_rosto = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

camera = CSICamera(width=300, height=300, capture_width=1280, capture_height=720, capture_fps=15)
imagem_widget = widgets.Image(format='jpeg', width=300, height=300)
botao_desligar = widgets.Button(description="❌ DESLIGAR", button_style='danger')

display(imagem_widget, botao_desligar)
sistema_ativo = True

def detetar_rosto(change):
    global sistema_ativo
    if not sistema_ativo: return
    
    frame = change['new']
    cinzento = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Executa a deteção de rostos na imagem cinzenta
    rostos = classificador_rosto.detectMultiScale(cinzento, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    
    for (x, y, largura, altura) in rostos:
        # Desenha o quadrado azul à volta do rosto detetado
        cv2.rectangle(frame, (x, y), (x + largura, y + altura), (255, 0, 0), 2)
        
        # --- 🎯 ESTIMATIVA DE DISTÂNCIA ---
        if largura > 120:
            cv2.putText(frame, "Muito Perto! TRAVAR", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
        else:
            cv2.putText(frame, "Alvo detetado. A seguir...", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            
    _, jpeg = cv2.imencode('.jpg', frame)
    imagem_widget.value = jpeg.tobytes()
    time.sleep(0.02)

camera.observe(detetar_rosto, names='value')

def encerra(b):
    global sistema_ativo
    sistema_ativo = False
    camera.unobserve(detetar_rosto, names='value')
    camera.running = False
    print("Sistema desligado.")

botao_desligar.on_click(encerra)
camera.running = True